# Chapter 43: Transfer Learning & Fine-Tuning
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo
**Objective:** Mempelajari strategi transfer representasi visual dari model terlatih (*pretrained model*), membedakan Feature Extraction vs Fine-Tuning, serta mengimplementasikan penjadwalan laju pembelajaran diskriminatif dengan PyTorch / NumPy.

### 1. Landasan Teori & Formulasi Matematika
Pembaruan bobot lapis demi lapis dengan Discriminative Learning Rates:
$$\eta_l = \eta_{\text{base}} \cdot \gamma^{(L - l)}, \quad \gamma \in (0, 1)$$

Penjadwalan Cosine Annealing dengan Linear Warmup:
$$\eta_t = \begin{cases}
\eta_{\min} + \frac{t}{T_{\text{warm}}} (\eta_{\max} - \eta_{\min}) & t \le T_{\text{warm}} \\
\eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min}) \left(1 + \cos\left(\frac{t - T_{\text{warm}}}{T_{\max} - T_{\text{warm}}} \pi\right)\right) & t > T_{\text{warm}}
\end{cases}$$

Pencegahan Gradient Shock pada Batch Normalization saat fine-tuning:
$$\hat{\mathbf{x}} = \frac{\mathbf{x} - \boldsymbol{\mu}_{\text{pretrained}}}{\sqrt{\boldsymbol{\sigma}^2_{\text{pretrained}} + \epsilon}} \odot \boldsymbol{\gamma} + \boldsymbol{\beta}$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import math
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    import torch.nn as nn
    import torchvision.models as models
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Menjalankan engine optimasi deterministik.")

In [ ]:
# 3. Implementasi Cosine Annealing Scheduler with Warmup
def get_lr(step, total_steps=100, warmup_steps=10, lr_min=1e-6, lr_max=1e-3):
    if step <= warmup_steps:
        return lr_min + (step / warmup_steps) * (lr_max - lr_min)
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return lr_min + 0.5 * (lr_max - lr_min) * (1.0 + math.cos(math.pi * progress))

steps = [0, 5, 10, 25, 50, 75, 100]
lrs = [get_lr(s) for s in steps]
for s, lr in zip(steps, lrs):
    print(f"Step {s:3d}: LR = {lr:.6f}")
assert np.isclose(lrs[0], 1e-6)
assert np.isclose(lrs[2], 1e-3)
assert np.isclose(lrs[-1], 1e-6)

In [ ]:
# 4. Eksperimen Discriminative Learning Rates & Drift Bobot
base_lr = 0.001
decay = 0.1
num_layers = 3

layer_lrs = [base_lr * (decay ** (num_layers - 1 - i)) for i in range(num_layers)]
weights = [0.5000, -0.3000, 0.1000]
grads = [1.5, 2.0, 4.0]

new_weights = [w - lr * g for w, lr, g in zip(weights, layer_lrs, grads)]

print("Learning Rates per Lapisan (Bottom -> Top):", layer_lrs)
print("Bobot Awal:     ", weights)
print("Bobot Terbarukan:", [round(w, 6) for w in new_weights])

drifts = [abs(nw - w) / abs(w) * 100 for nw, w in zip(new_weights, weights)]
print(f"Pergeseran Bobot (%): {[round(d, 4) for d in drifts]}")
assert drifts[-1] > 100 * drifts[0], "Head harus bergeser jauh lebih cepat daripada representasi awal"

In [ ]:
# 5. Simulasi Pembekuan Backbone (Feature Extraction Mode)
def simulate_feature_extraction(x_features, y_target, epochs=20, lr=0.01):
    np.random.seed(42)
    w_head = np.random.randn(8, 1) * 0.1
    b_head = 0.0
    for ep in range(epochs):
        logits = np.dot(x_features, w_head) + b_head
        probs = 1.0 / (1.0 + np.exp(-np.clip(logits, -15, 15)))
        loss = -np.mean(y_target * np.log(probs + 1e-12) + (1 - y_target) * np.log(1 - probs + 1e-12))
        grad_logits = (probs - y_target) / len(y_target)
        grad_w = np.dot(x_features.T, grad_logits)
        grad_b = np.sum(grad_logits)
        w_head -= lr * grad_w
        b_head -= lr * grad_b
    return w_head, b_head, loss

x_mock = np.random.randn(50, 8)
y_mock = (x_mock[:, 0] > 0).astype(float).reshape(-1, 1)
w_final, b_final, final_loss = simulate_feature_extraction(x_mock, y_mock)
print(f"Final Feature Extraction Head Loss: {final_loss:.4f}")
assert final_loss < 0.693

In [ ]:
# 6. Real PyTorch Pretrained Model Adaptation (MobileNetV3 / ResNet-18)
if HAS_TORCH:
    # 1. Load Pretrained MobileNetV3-Small (Ringan & Lisensi BSD-3)
    mobilenet = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    print("Pretrained MobileNetV3-Small Loaded Successfully.")
    
    # 2. Bekukan seluruh parameter backbone (Feature Extraction Mode)
    for param in mobilenet.features.parameters():
        param.requires_grad = False
        
    # 3. Ganti Linear Head untuk klasifikasi 2 kelas target kustom
    in_features = mobilenet.classifier[3].in_features
    mobilenet.classifier[3] = nn.Linear(in_features, 2)
    
    trainable_p = sum(p.numel() for p in mobilenet.parameters() if p.requires_grad)
    total_p = sum(p.numel() for p in mobilenet.parameters())
    print(f"Total Parameter: {total_p:,} | Trainable Parameter: {trainable_p:,}")
    print(f"Rasio Bobot yang Dilatih: {(trainable_p / total_p)*100:.2f}%")
else:
    print("Panduan Arsitektur Pretrained PyTorch Transfer Learning:")
    print("1. Pretrained Backbone: torchvision.models.mobilenet_v3_small(weights=DEFAULT)")
    print("2. Pembekuan Bobot: Set 'param.requires_grad = False' pada seluruh blok fitur.")
    print("3. Head Replacement: Ganti layer terakhir 'classifier[3]' dengan Linear(in_feat, num_classes).")
    print("4. Rasio efisiensi: Hanya ~2.000 dari 2.540.000 parameter (<0.1%) yang dilatih pada tahap awal!")

### 7. Analisis Hasil & Pedoman Praktik Riset
1. **Protokol Dua Tahap (Two-Stage Pipeline)**: Selalu latih classifier head terlebih dahulu selama 5 epoch sebelum membuka blok konvolusi atas untuk fine-tuning guna menghindari *gradient shock*.
2. **Freeze Running Stats pada Batch Normalization**: Saat fine-tuning dengan batch mini kecil, pertahankan BN dalam mode evaluasi agar mean dan varians ImageNet populasi asli tidak rusak.
3. **Efisiensi Memori**: Feature extraction memungkinkan pemanfaatan model visi raksasa pada perangkat edge dan laptop standar dengan konsumsi daya minimal.